In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.metrics import accuracy_score

# 1. Load data
df = pd.read_csv('cbc_final_data.csv')

# 2. Filter Females and Drop unnecessary columns
# We drop 'weight_kg' and 'Gender' and 'Diagnosis' to keep only blood metrics
df_female = df[df['Gender'] == 0].copy()
df_female = df_female.drop(['Gender', 'Diagnosis'], axis=1, errors='ignore')

# If weight_kg exists in your CSV, drop it here to be safe
if 'weight_kg' in df_female.columns:
    df_female = df_female.drop('weight_kg', axis=1)

print(f"Data ready. Records: {len(df_female)}")

Data ready. Records: 3426


In [2]:
labels = [
    'Leukopenia', 'Leukocytosis', 'Anemia', 'Elevated Hemoglobin',
    'Thrombocytopenia', 'Thrombocytosis', 'Microcytosis', 'Macrocytosis',
    'Elevated RDW', 'Neutropenia', 'Neutrophilia', 'Lymphopenia', 'Lymphocytosis',
    'Monocytopenia', 'Monocytosis', 'Eosinophilia', 'Basophilia',
    'Probable Iron Deficiency Anemia (IDA)'
]

def apply_rules(row):
    d = []
    if row['WBC'] < 4.0: d.append('Leukopenia')
    if row['WBC'] > 10.0: d.append('Leukocytosis')
    if row['Hb'] < 12.0: d.append('Anemia')
    if row['Hb'] > 15.5: d.append('Elevated Hemoglobin')
    if row['PLATELETS'] < 150: d.append('Thrombocytopenia')
    if row['PLATELETS'] > 450: d.append('Thrombocytosis')
    if row['MCV'] < 80: d.append('Microcytosis')
    if row['MCV'] > 100: d.append('Macrocytosis')
    if row['RDW'] > 15.0: d.append('Elevated RDW')
    if row['MCV'] < 80.0 and row['RDW'] > 15.0:
        d.append('Probable Iron Deficiency Anemia (IDA)')
    return d

y_list = df_female.apply(apply_rules, axis=1)
mlb = MultiLabelBinarizer(classes=labels)
y = mlb.fit_transform(y_list)

In [3]:
# Exact feature list for the API
features = ['Age', 'Hb', 'RBC', 'WBC', 'PLATELETS', 'HCT', 'MCV', 'MCH', 'MCHC',
            'RDW', 'PDW', 'MPV', 'PCT', 'Lymphocytes', 'Monocytes', 'Neutrophils',
            'Eosinophils', 'Basophils']

X = df_female[features]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train
rf_model = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
rf_model.fit(X_train_scaled, y_train)

print("Model trained successfully without weight_kg!")

Model trained successfully without weight_kg!


In [4]:
# Save everything to be used by main.py
joblib.dump(rf_model, 'female_cbc_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(mlb, 'mlb.pkl')
joblib.dump(features, 'features_list.pkl')

print("All files saved! You can now restart your API.")

All files saved! You can now restart your API.
